# Example 4.5 / 1: Slices Through a Pair Set

Companion notebook for **4.5 Sets of Pairs** in the AMPL lab Documentation, section "Slices Through a Pair Set". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Slices Through a Pair Set

`{ORIGINS, DESTINATIONS}` would have all 2 × 3 = 6 arrows; `LINKS` has only the 4 that exist,
so `Ship` has 4 variables, not 6.

When one component of a pair is already known, the tuple expression selects a slice. In
`Supply`, the constraint's own index fixes `i`, so `{(i,j) in LINKS}` runs only over the
destinations linked to that origin; in `Demand`, `j` is fixed and the same expression runs over
the origins linked to that destination:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIGINS;                               # supply points
set DESTINATIONS;                          # demand points
set LINKS within {ORIGINS, DESTINATIONS};  # shipping links that exist

param supply {ORIGINS} >= 0;       # units available at each origin
param demand {DESTINATIONS} >= 0;  # units required at each destination
param cost {LINKS} >= 0;           # cost per unit shipped on each link

var Ship {LINKS} >= 0;             # units shipped on each link

minimize Total_Cost: sum {(i,j) in LINKS} cost[i,j] * Ship[i,j];

subject to Supply {i in ORIGINS}:      # an origin ships at most its supply
    sum {(i,j) in LINKS} Ship[i,j] <= supply[i];

subject to Demand {j in DESTINATIONS}: # a destination receives at least its demand
    sum {(i,j) in LINKS} Ship[i,j] >= demand[j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIGINS"] = ["GARY", "CLEV"]
ampl.set["DESTINATIONS"] = ["DET", "LAN", "STL"]
ampl.set["LINKS"] = [("GARY", "DET"), ("GARY", "STL"), ("CLEV", "LAN"), ("CLEV", "DET")]
ampl.param["supply"] = {"GARY": 1400, "CLEV": 1500}
ampl.param["demand"] = {"DET": 1100, "LAN": 900, "STL": 500}
ampl.param["cost"] = {("GARY", "DET"): 14, ("GARY", "STL"): 16,
                      ("CLEV", "LAN"): 12, ("CLEV", "DET"): 9}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["Total_Cost"].value())
print(ampl.var["Ship"].to_pandas())

solved 31200.0
               Ship.val
index0 index1          
CLEV   DET          600
       LAN          900
GARY   DET          500
       STL          500


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set ORIGINS := GARY CLEV;
set DESTINATIONS := DET LAN STL;
set LINKS := (GARY,DET) (GARY,STL) (CLEV,LAN) (CLEV,DET);
param supply := GARY 1400  CLEV 1500;
param demand := DET 1100  LAN 900  STL 500;
param cost := GARY DET 14  GARY STL 16  CLEV LAN 12  CLEV DET 9;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Total_Cost, Ship;

HiGHS 1.14.0: optimal solution; objective 31200
0 simplex iterations
0 barrier iterations
Total_Cost = 31200

Ship :=
CLEV DET   600
CLEV LAN   900
GARY DET   500
GARY STL   500
;

